# Final Project

本 Notebook 与当前脚本职责保持一致：
- `Final_Project.py`：只负责生成提交文件 `outputs/submission.csv`。
- `Final_Project_Eval.py`：独立负责离线评估与调参。

## 1. 导入依赖与环境检查

In [ ]:
from pathlib import Path

from Final_Project import (
    BASE_PATH,
    OUTPUT_DIR,
    print_environment_info,
    print_data_file_status,
    load_tables,
    summarize_inputs,
    prepare_transactions,
    prepare_article_department,
    load_ranker_from_cache,
    describe_ranker,
    generate_submission,
    run_pipeline,
)

print_environment_info()
print_data_file_status(BASE_PATH)

## 2. 查看输入数据概览

In [ ]:
tables = load_tables(BASE_PATH)
summary = summarize_inputs(tables)
summary

## 3. 生成层：仅产出 submission.csv

这一步是“手动分步版”，和 `run_pipeline` 执行逻辑一致。

In [ ]:
transactions = prepare_transactions(tables["transactions"])
article_department = prepare_article_department(tables["articles"])

ranker_config = load_ranker_from_cache(output_dir=OUTPUT_DIR)
print("Selected ranker:", describe_ranker(ranker_config))

submission = generate_submission(
    tables=tables,
    transactions=transactions,
    article_department=article_department,
    ranker_config=ranker_config,
    output_dir=OUTPUT_DIR,
)

submission.head(5)

## 4. 检查输出路径

In [ ]:
output_path = Path(OUTPUT_DIR) / "submission.csv"
print(output_path)
print("exists:", output_path.exists())

## 5. 生成层：一键运行版

如需要一条命令跑完整生成流程，可使用下面入口。

In [ ]:
state = run_pipeline(BASE_PATH)
state["submission"].head(5)

## 6. 评估层（可选，单独执行）

评估与调参已拆分到 `Final_Project_Eval.py`，避免影响生成层速度。

如需要运行评估，建议在单独会话中执行：
```bash
python Final_Project_Eval.py
```

In [ ]:
from Final_Project_Eval import run_eval_pipeline

# 可选：这一步计算耗时较长，默认注释
# eval_state = run_eval_pipeline(BASE_PATH)
# eval_state["fold_metrics"]